# 07 — Visualizations: Global Building Dataset Validation

In [20]:
from google.colab import drive
import os
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
else:
    print('Drive already mounted.')

Drive already mounted.


In [21]:
# Install plotting dependencies
import subprocess, sys
# kaleido 0.2.1 uses a bundled renderer — no Chrome needed, reliable in Colab
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "plotly", "kaleido==0.2.1"], check=False)

# Download Barlow font (optional — falls back to Arial if not available)
import os
try:
    os.system("wget -q -O /content/Barlow-Regular.ttf https://github.com/jpt/barlow/raw/main/fonts/ttf/Barlow-Regular.ttf || true")
    os.system("wget -q -O /content/BarlowSemiCondensed-Regular.ttf https://github.com/jpt/barlow/raw/main/fonts/ttf/BarlowSemiCondensed-Regular.ttf || true")
    import matplotlib.font_manager as _fm
    _fm.fontManager.addfont('/content/Barlow-Regular.ttf')
    _fm.fontManager.addfont('/content/BarlowSemiCondensed-Regular.ttf')
except Exception:
    pass

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

FONT_FAMILY = 'Barlow, Arial, sans-serif'
FONT_COLOR  = '#1a1a1a'
BG_COLOR    = 'white'

DATASET_COLORS = {
    'overture':               '#1B6CA8',
    'gba':                    '#2BAE82',
    'globfp':                 '#66C7E0',
    'wsf_tracker@10m':        '#C94A35',
    'obt_2023@10m':           '#E0882A',
    'ghsl_built_s_2025@100m': '#CC2266',
    'tempo_2023q4@100m':      '#7B4EBD',
    'wsf_tracker@100m':       '#EFB3A9',
    'obt_2023@100m':          '#F5C98A',
}
DATASET_LABELS = {
    'overture':               'Overture Maps',
    'gba':                    'Google Open Buildings',
    'globfp':                 'GlobFP',
    'wsf_tracker@10m':        'WSF Tracker (10m)',
    'obt_2023@10m':           'OBT 2023 (10m)',
    'ghsl_built_s_2025@100m': 'GHSL 2025 (100m)',
    'tempo_2023q4@100m':      'TEMPO Q4 2023 (100m)',
    'wsf_tracker@100m':       'WSF Tracker (100m)',
    'obt_2023@100m':          'OBT 2023 (100m)',
}
SOURCE_COLORS = {
    'SpaceNet7':   '#E05C00',
    'HotOSM':      '#0072B2',
    'Other / Gov': '#009E73',
}

def apply_ppt_style(fig, title=None, height=720, width=1280):
    fig.update_layout(
        font=dict(family=FONT_FAMILY, color=FONT_COLOR, size=15),
        plot_bgcolor=BG_COLOR,
        paper_bgcolor=BG_COLOR,
        height=height,
        width=width,
        title_font=dict(size=20, family=FONT_FAMILY, color=FONT_COLOR),
        title_x=0.5,
        title_xanchor='center',
        legend=dict(
            font=dict(size=13, family=FONT_FAMILY),
            bgcolor='rgba(255,255,255,0.9)',
            bordercolor='#CCCCCC',
            borderwidth=1,
        ),
    )
    fig.update_xaxes(
        title_font=dict(size=15, family=FONT_FAMILY),
        tickfont=dict(size=13, family=FONT_FAMILY),
        gridcolor='#E8E8E8',
    )
    fig.update_yaxes(
        title_font=dict(size=15, family=FONT_FAMILY),
        tickfont=dict(size=13, family=FONT_FAMILY),
        gridcolor='#E8E8E8',
    )
    if title is not None:
        fig.update_layout(title_text=title)
    return fig

print('Setup complete.')


Setup complete.


In [22]:
# ── Only edit this cell ────────────────────────────────────────────────────
DATA_DIR = '/content/drive/MyDrive/WorldBank/FY26 - DEP/Gates Foundation/Building Dataset Validation/outputs/global_metrics/'   # ← set to folder containing your CSVs
# ──────────────────────────────────────────────────────────────────────────
OUT_DIR  = '/content/drive/MyDrive/WorldBank/FY26 - DEP/Gates Foundation/Building Dataset Validation/outputs/figures/'
SENS_DIR = os.path.dirname(DATA_DIR.rstrip('/')) + '/sensitivity_studies/'
os.makedirs(OUT_DIR, exist_ok=True)
print(f'Figures will be saved to: {OUT_DIR}')


Figures will be saved to: /content/drive/MyDrive/WorldBank/FY26 - DEP/Gates Foundation/Building Dataset Validation/outputs/figures/


In [44]:
vec = pd.read_csv(DATA_DIR + 'vector_all_cities_merged.csv')
ras = pd.read_csv(DATA_DIR + 'raster_all_cities_merged.csv')

# IoU threshold sensitivity (may not exist yet)
try:
    iou = pd.read_csv(SENS_DIR + 'iou_threshold_sensitivity.csv')
    print(f'iou: {len(iou)} rows')
except FileNotFoundError:
    iou = pd.DataFrame()
    print('iou: file not found — S1/S2 figures will be skipped')

# WSF year-sensitivity outputs
try:
    wsf_sweep   = pd.read_csv(SENS_DIR + 'wsf_year_sensitivity_summary.csv')
    wsf_aligned = pd.read_csv(SENS_DIR + 'wsf_temporal_alignment.csv')
    print(f'wsf_sweep: {len(wsf_sweep)} rows  |  wsf_aligned: {len(wsf_aligned)} rows')
except FileNotFoundError:
    wsf_sweep = pd.DataFrame()
    wsf_aligned = pd.DataFrame()
    print('WSF sensitivity CSVs not found — W1/W2 figures will be skipped')

# Build raster ds_key
ras['ds_key'] = ras['dataset'].astype(str).str.strip().str.lower() + '@' + ras['grid'].astype(str).str.strip().str.lower()

# Reference source classification for vector data
SN7 = {
    'usa-brentwood', 'usa-sandiego', 'usa-boise', 'usa-lasvegas', 'usa-saltlakecity',
    'usa-permianbasin', 'mex-mexicocity', 'usa-bentonville', 'usa-gonzales', 'usa-atlanta',
    'usa-portstlucie', 'pan-panamacity', 'usa-raleigh', 'jam-kingston', 'usa-allentown',
    'per-cusco', 'chl-calama', 'bra-manaus', 'bra-saopaulo', 'sen-dakar', 'alg-tindouf',
    'gha-kumasi', 'lby-benghazi', 'zmb-lusaka', 'zaf-durban', 'egy-cairo', 'sdn-khartoum',
    'uga-kampala', 'gbr-birmingham', 'gbr-london', 'nld-rotterdam', 'rou-bucharest',
    'yem-dhamar', 'sau-riyadh', 'kwt-kuwaitcity', 'rus-astrakhan', 'are-abudhabi',
    'uzb-zarafshan', 'ind-mumbai', 'ind-tada', 'ind-vijayawada', 'bgd-dhaka', 'chn-chengdu',
    'chn-zhuhai', 'chn-guangzhou', 'chn-wuhan', 'chn-lujiang', 'chn-yangzhou',
    'phl-angelescity', 'chn-shanghai', 'kor-sejong', 'aus-melbourne'
}
HOT = {
    'ant-curacao','bgd-rohingya','blz-burrell-boom','bra-nova-sussuarana',
    'col-san-antonio-de-prado','gha-accra','gha-aiyim-sraha','gha-dansoman','gha-nawuni',
    'gha-sawla-tuna','gha-wa','jam-kingston','jam-saint-catherine','jpn-ashiya-hama',
    'jpn-hiroshima','jpn-iwate','jpn-izu-oshima','jpn-kashima','jpn-numakunai',
    'ken-kakuma','ken-kakuma-kalobeyei','ken-mukuru','lbr-monrovia','lby-almarj',
    'lby-bayda','lby-darnah','lby-susah','maf-saint-martin','mmr-patheingyi-mandalay',
    'moz-de-maio','moz-djonasse','mwi-lilongwe','mwi-mlowe','ner-niame','nga-ibadan',
    'phl-bagamanoc','phl-barangay','phl-catanduanes','phl-juraojurao-anini-y-antique',
    'phl-pasig','phl-poblacion-sagua-anini-y-antique','phl-viga','phl-visayas',
    'sle-cockle-bay-1','sle-cockle-bay-2','sle-freetown','sle-kolleh','sle-kroo-bay-1',
    'swz-nhlangano','sxm-sint-maarten','tjk-artuch','tjk-nomandiyon','tjk-tavishi-bolo',
    'ton-nukualofa','ton-sopu','ton-tatakamotonga','ton-tokomololo','tto-la-brea',
    'tto-sangre','uga-bugoye','uga-kanara','uga-nakamiro','ukr-pulyny',
} # to-do verify this overview of cities!!
vec['source'] = vec['city'].apply(
    lambda c: 'SpaceNet7' if c in SN7 else ('HotOSM' if c in HOT else 'Other')
)

# Numeric coercion
for _c in ['f1_city','precision_city','recall_city','total_area_bias',
           'count_ratio_total','boundary_f_meanpair_tp']:
    if _c in vec.columns:
        vec[_c] = pd.to_numeric(vec[_c], errors='coerce')

for _c in ['f1_tile_mean','signed_area_bias','rel_area_error_mean']:
    if _c in ras.columns:
        ras[_c] = pd.to_numeric(ras[_c], errors='coerce')

print(f'vec: {len(vec)} rows, {vec["dataset"].nunique()} datasets, {vec["city"].nunique()} cities')
print(f'ras: {len(ras)} rows, {ras["ds_key"].nunique()} ds_keys, {ras["city"].nunique()} cities')
print(f'vec datasets: {sorted(vec["dataset"].unique())}')
print(f'ras ds_keys:  {sorted(ras["ds_key"].unique())}')


iou: file not found — S1/S2 figures will be skipped
WSF sensitivity CSVs not found — W1/W2 figures will be skipped
vec: 400 rows, 3 datasets, 135 cities
ras: 722 rows, 6 ds_keys, 136 cities
vec datasets: ['gba', 'globfp', 'overture']
ras ds_keys:  ['ghsl_built_s_2025@100m', 'obt_2023@100m', 'obt_2023@10m', 'tempo_2023q4@100m', 'wsf_tracker@100m', 'wsf_tracker@10m']


In [ ]:
# ── Zero-F1 city exclusion (item 8b) ─────────────────────────────────────────
# Exclude cities where ALL datasets report F1 = 0. These are coverage failures or
# pipeline errors, NOT genuine zero-accuracy results, and they distort global means
# and error bars. Vector and raster are filtered SEPARATELY, each with its own
# criterion. Filtering is in-memory only — the raw CSV files are NOT modified.

# --- Vector: exclude a city if f1_city == 0 for overture AND gba AND globfp ---
VECTOR_FILTER_DATASETS = ['overture', 'gba', 'globfp']
_vpiv = (vec.assign(_ds=vec['dataset'].astype(str).str.strip().str.lower())
            .drop_duplicates(['city', '_ds'])
            .pivot(index='city', columns='_ds', values='f1_city')
            .reindex(columns=VECTOR_FILTER_DATASETS))
# A city is excluded only if all three datasets are present and every one is 0.
# (A missing dataset -> NaN -> NaN == 0 is False -> the city is kept.)
excluded_vec = sorted(_vpiv.index[(_vpiv == 0).all(axis=1)].tolist())
if excluded_vec:
    print(f"[vector] Excluding {len(excluded_vec)} cities (all datasets F1=0): {excluded_vec}")
else:
    print("[vector] No cities excluded (no city has all datasets F1=0).")
vec = vec[~vec['city'].isin(excluded_vec)]
print(f"[vector] Cities remaining: {vec['city'].nunique()}  ({len(vec)} rows)\n")

# --- Raster: exclude a city if f1_tile_mean == 0 for all 6 ds_key values ---
RASTER_FILTER_DSKEYS = [
    'wsf_tracker@10m', 'obt_2023@10m', 'ghsl_built_s_2025@100m',
    'tempo_2023q4@100m', 'wsf_tracker@100m', 'obt_2023@100m',
]
_rpiv = (ras.drop_duplicates(['city', 'ds_key'])
            .pivot(index='city', columns='ds_key', values='f1_tile_mean')
            .reindex(columns=RASTER_FILTER_DSKEYS))
excluded_ras = sorted(_rpiv.index[(_rpiv == 0).all(axis=1)].tolist())
if excluded_ras:
    print(f"[raster] Excluding {len(excluded_ras)} cities (all datasets F1=0): {excluded_ras}")
else:
    print("[raster] No cities excluded (no city has all ds_keys F1=0).")
ras = ras[~ras['city'].isin(excluded_ras)]
print(f"[raster] Cities remaining: {ras['city'].nunique()}  ({len(ras)} rows)")

In [39]:
# ── V1: Overall macro F1 — vector ─────────────────────────────────────────
VEC_DS = ['overture', 'gba', 'globfp']
_v = vec[vec['dataset'].isin(VEC_DS)].copy()

fig_v1 = go.Figure()

for ds in VEC_DS:
    _d = _v[_v['dataset'] == ds]
    fig_v1.add_trace(go.Violin(
        y=_d['f1_city'],
        name=DATASET_LABELS.get(ds, ds),
        box_visible=True,
        meanline_visible=True,
        line_color=DATASET_COLORS.get(ds, '#888'),
        fillcolor=DATASET_COLORS.get(ds, '#888'),
        opacity=0.6,
        marker_color=DATASET_COLORS.get(ds, '#888'),
        showlegend=False,
        points='all',
        jitter=0.2,
        text=_d['city'].tolist(),
        hovertemplate='<b>%{text}</b><br>F1: %{y:.3f}<extra></extra>'
    ))

apply_ppt_style(fig_v1,
    title='Vector Datasets — Overall Accuracy (Macro F1 distribution across cities)',
    height=500, width=900)
fig_v1.update_layout(yaxis=dict(range=[0, 1.15], title='Macro F1'), xaxis_title='Dataset')
fig_v1.write_image(os.path.join(OUT_DIR, 'V1_vector_macro_f1.png'), scale=2)
apply_ppt_style(fig_v1)
fig_v1.show()
print('Saved: V1_vector_macro_f1.png')

/tmp/ipykernel_8872/3343496494.py:29: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig_v1.write_image(os.path.join(OUT_DIR, 'V1_vector_macro_f1.png'), scale=2)


Saved: V1_vector_macro_f1.png


In [49]:
# ── V2: F1 by reference data source ───────────────────────────────────────
import math

_v2 = vec[vec['dataset'].isin(VEC_DS)].copy()
_sources = ['SpaceNet7', 'HotOSM', 'Other']

fig_v2 = go.Figure()

shown_in_legend = set()

for ds in VEC_DS: # Iterate through datasets for grouping and coloring
    _d_all_sources = _v2[_v2['dataset'] == ds]
    ds_label = DATASET_LABELS.get(ds, ds)

    for src in _sources: # Iterate through sources to set x-axis order
        _d_filtered = _d_all_sources[_d_all_sources['source'] == src]

        if not _d_filtered.empty:
            fig_v2.add_trace(go.Violin(
                y=_d_filtered['f1_city'],
                x=[src] * len(_d_filtered['f1_city']), # X-axis will be data source
                name=ds_label, # Name will be dataset for grouping and legend
                legendgroup=ds_label, # Group legends by dataset
                showlegend=ds_label not in shown_in_legend, # Show legend only once per dataset
                box_visible=True,
                meanline_visible=True,
                line_color=DATASET_COLORS.get(ds, '#888'),
                fillcolor=DATASET_COLORS.get(ds, '#888'),
                opacity=0.6,
                marker_color=DATASET_COLORS.get(ds, '#888'),
                points='all', # Show individual points
                jitter=0.2,
                text=_d_filtered['city'].tolist(),
                hovertemplate='<b>%{text}</b><br>F1: %{y:.3f}<extra></extra>'
            ))
            shown_in_legend.add(ds_label)

apply_ppt_style(fig_v2,
    title='Vector F1 Distribution by Reference Data Source',
    height=650, width=1100)
fig_v2.update_layout(
    violinmode='group', # Group violins by x-axis categories
    violingap=0.1,      # Reduce whitespace between categories
    violingroupgap=0.1, # Reduce whitespace between violins within a category
    yaxis=dict(title='F1 (city level)', range=[0, 1.0]),
    xaxis_title='Reference Data Source',
    xaxis=dict(categoryorder='array', categoryarray=_sources) # Ensure order of sources on x-axis
)
fig_v2.write_image(os.path.join(OUT_DIR, 'V2_f1_by_source.png'), scale=2)
apply_ppt_style(fig_v2)
fig_v2.show()
print('Saved: V2_f1_by_source.png')

/tmp/ipykernel_8872/560568979.py:49: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig_v2.write_image(os.path.join(OUT_DIR, 'V2_f1_by_source.png'), scale=2)


Saved: V2_f1_by_source.png


In [50]:
# ── V3: Dataset wins — vector ─────────────────────────────────────────────
_v3_cov = vec[vec['dataset'].isin(VEC_DS)].groupby('city')['dataset'].nunique()
_cities_all3 = _v3_cov[_v3_cov == 3].index
vec3 = vec[vec['city'].isin(_cities_all3) & vec['dataset'].isin(VEC_DS)].copy()
print(f'Cities with all 3 vector datasets: {len(_cities_all3)}')

_rank_counts = {ds: {1: 0, 2: 0, 3: 0} for ds in VEC_DS}
for city, grp in vec3.groupby('city'):
    ranked = grp.sort_values('f1_city', ascending=False).reset_index(drop=True)
    for rank_idx, row in ranked.iterrows():
        ds = row['dataset']
        if ds in _rank_counts:
            _rank_counts[ds][rank_idx + 1] = _rank_counts[ds].get(rank_idx + 1, 0) + 1

fig_v3 = go.Figure()
_opacities = {1: 1.0, 2: 0.55, 3: 0.25}
for rank in [3, 2, 1]:
    fig_v3.add_trace(go.Bar(
        y=[DATASET_LABELS.get(d, d) for d in VEC_DS],
        x=[_rank_counts[d].get(rank, 0) for d in VEC_DS],
        name=f'Rank {rank}',
        orientation='h',
        marker_color=[DATASET_COLORS.get(d, '#888') for d in VEC_DS],
        opacity=_opacities[rank],
    ))
for i, ds in enumerate(VEC_DS):
    r1 = _rank_counts[ds].get(1, 0)
    fig_v3.add_annotation(
        y=DATASET_LABELS.get(ds, ds), x=len(_cities_all3) + 2,
        text=f'{r1} wins', showarrow=False,
        font=dict(size=12, family=FONT_FAMILY, color=DATASET_COLORS.get(ds, '#333')),
    )
apply_ppt_style(fig_v3,
    title=f'Which Vector Dataset Wins? (n={len(_cities_all3)} cities, all datasets present)',
    height=400, width=900)
fig_v3.update_layout(barmode='stack', xaxis_title='Number of Cities', yaxis_title='')
fig_v3.write_image(os.path.join(OUT_DIR, 'V3_vector_wins.png'), scale=2)
apply_ppt_style(fig_v3)
fig_v3.show()
print('Saved: V3_vector_wins.png')


Cities with all 3 vector datasets: 130


/tmp/ipykernel_8872/2517498023.py:37: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig_v3.write_image(os.path.join(OUT_DIR, 'V3_vector_wins.png'), scale=2)


Saved: V3_vector_wins.png


In [52]:
# ── V4: Complementary metrics — vector ────────────────────────────────────
_v4 = vec[vec['dataset'].isin(VEC_DS)].copy()
_metrics_v4 = [
    ('f1_city',              'Macro F1',               [0, 1.15],   None),
    ('total_area_bias',  'Total Area Bias',       None,        0),
    ('count_ratio_total',    'Count Ratio',            None,        1.0),
]

fig_v4 = make_subplots(rows=1, cols=3,
    subplot_titles=[m[1] for m in _metrics_v4],
    shared_yaxes=True, horizontal_spacing=0.06)

for col_idx, (col, label, x_range, vline) in enumerate(_metrics_v4, 1):
    if col not in _v4.columns:
        print(f'[V4] missing column: {col}')
        continue

    for ds in VEC_DS:
        _d = _v4[_v4['dataset'] == ds].dropna(subset=[col])
        fig_v4.add_trace(go.Box(
            x=_d[col],
            y=[DATASET_LABELS.get(ds, ds)] * len(_d),
            name=DATASET_LABELS.get(ds, ds),
            orientation='h',
            marker_color=DATASET_COLORS.get(ds, '#888'),
            showlegend=False,
        ), row=1, col=col_idx)

    if vline is not None:
        fig_v4.add_vline(x=vline, line_dash='dash', line_color='gray', opacity=0.5, row=1, col=col_idx)
    if x_range:
        fig_v4.update_xaxes(range=x_range, row=1, col=col_idx)

apply_ppt_style(fig_v4,
    title='Vector Accuracy: F1, Area Bias, and Building Count Ratio',
    height=550, width=1400)
fig_v4.write_image(os.path.join(OUT_DIR, 'V4_complementary_metrics_vector.png'), scale=2)
apply_ppt_style(fig_v4)
fig_v4.show()
print('Saved: V4_complementary_metrics_vector.png')

/tmp/ipykernel_8872/1350379506.py:37: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig_v4.write_image(os.path.join(OUT_DIR, 'V4_complementary_metrics_vector.png'), scale=2)


Saved: V4_complementary_metrics_vector.png


In [28]:
# ── V5: F1 vs total area bias scatter ────────────────────────────────────
_v5 = vec[vec['dataset'].isin(VEC_DS)].dropna(subset=['total_area_bias', 'f1_city'])
fig_v5 = go.Figure()
for ds in VEC_DS:
    _d = _v5[_v5['dataset'] == ds]
    fig_v5.add_trace(go.Scatter(
        x=_d['total_area_bias'], y=_d['f1_city'],
        mode='markers', name=DATASET_LABELS.get(ds, ds),
        marker=dict(color=DATASET_COLORS.get(ds, '#888'), size=7, opacity=0.7),
        hovertemplate='<b>%{text}</b><br>Bias: %{x:.3f}<br>F1: %{y:.3f}<extra></extra>',
        text=_d['city'].tolist(),
    ))
fig_v5.add_vline(x=0, line_color='#333', line_width=1.5)
fig_v5.add_vline(x=0.10, line_dash='dash', line_color='gray', opacity=0.5)
fig_v5.add_vline(x=-0.10, line_dash='dash', line_color='gray', opacity=0.5)
apply_ppt_style(fig_v5, title='F1 vs Total Area Bias — Vector Datasets', height=650, width=1000)
fig_v5.update_layout(xaxis_title='Total Area Bias', yaxis_title='F1 (city level)')
fig_v5.write_image(os.path.join(OUT_DIR, 'V5_f1_vs_bias_vector.png'), scale=2)
apply_ppt_style(fig_v5)
fig_v5.show()
print('Saved: V5_f1_vs_bias_vector.png')

# ── V6: Precision vs Recall scatter ───────────────────────────────────────
_v6 = vec[vec['dataset'].isin(VEC_DS)].dropna(subset=['precision_city', 'recall_city'])
fig_v6 = go.Figure()
# F1 iso-curves
for f1_iso in [0.25, 0.50, 0.75]:
    _p = np.linspace(0.01, 1, 200)
    _r = f1_iso * _p / (2 * _p - f1_iso)
    _mask = (_r >= 0) & (_r <= 1)
    fig_v6.add_trace(go.Scatter(
        x=_p[_mask], y=_r[_mask], mode='lines',
        line=dict(color='#AAAAAA', dash='dot', width=1),
        showlegend=False,
        hoverinfo='skip',
    ))
    _mid = len(_p[_mask]) // 2
    if _mid < len(_p[_mask]):
        fig_v6.add_annotation(
            x=float(_p[_mask][_mid]), y=float(_r[_mask][_mid]),
            text=f'F1={f1_iso}', showarrow=False,
            font=dict(size=10, family=FONT_FAMILY, color='#999'),
        )
for ds in VEC_DS:
    _d = _v6[_v6['dataset'] == ds]
    fig_v6.add_trace(go.Scatter(
        x=_d['precision_city'], y=_d['recall_city'],
        mode='markers', name=DATASET_LABELS.get(ds, ds),
        marker=dict(color=DATASET_COLORS.get(ds, '#888'), size=7, opacity=0.7),
        hovertemplate='<b>%{text}</b><br>Prec: %{x:.3f}<br>Rec: %{y:.3f}<extra></extra>',
        text=_d['city'].tolist(),
    ))
apply_ppt_style(fig_v6, title='Precision vs Recall — Vector Datasets (city-level)', height=750, width=900)
fig_v6.update_layout(
    xaxis=dict(title='Precision', range=[0, 1.05]),
    yaxis=dict(title='Recall', range=[0, 1.05]),
)
fig_v6.write_image(os.path.join(OUT_DIR, 'V6_precision_recall.png'), scale=2)
apply_ppt_style(fig_v6)
fig_v6.show()
print('Saved: V6_precision_recall.png')


/tmp/ipykernel_8872/2278830762.py:18: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig_v5.write_image(os.path.join(OUT_DIR, 'V5_f1_vs_bias_vector.png'), scale=2)


Saved: V5_f1_vs_bias_vector.png


/tmp/ipykernel_8872/2278830762.py:58: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig_v6.write_image(os.path.join(OUT_DIR, 'V6_precision_recall.png'), scale=2)


Saved: V6_precision_recall.png


In [29]:
# ── V7: Boundary F-score ──────────────────────────────────────────────────
if 'boundary_f_meanpair_tp' not in vec.columns:
    print('[V7] boundary_f_meanpair_tp not in vec — skipping')
else:
    _v7 = vec[vec['dataset'].isin(VEC_DS)].copy()
    _stats7 = (_v7.groupby('dataset')['boundary_f_meanpair_tp']
                .agg(mean='mean', sd='std').reindex(VEC_DS)
                .sort_values('mean', ascending=True).reset_index())
    fig_v7 = go.Figure()
    fig_v7.add_trace(go.Bar(
        y=[DATASET_LABELS.get(d, d) for d in _stats7['dataset']],
        x=_stats7['mean'],
        orientation='h',
        marker_color=[DATASET_COLORS.get(d, '#888') for d in _stats7['dataset']],
        error_x=dict(type='data', array=_stats7['sd'].fillna(0).tolist(), visible=True, thickness=2),
        text=[f'<b>{v:.3f}</b>' for v in _stats7['mean']],
        textposition='outside',
        textfont=dict(size=13, family=FONT_FAMILY, color=FONT_COLOR),
        cliponaxis=False,
    ))
    apply_ppt_style(fig_v7,
        title='Boundary Delineation Accuracy — Vector Datasets',
        height=500, width=900)
    fig_v7.update_layout(
        xaxis=dict(range=[0, 1.15], title='Boundary F-score (mean)'),
        yaxis_title='',
        annotations=[dict(
            x=0.5, y=-0.15, xref='paper', yref='paper', showarrow=False,
            text='Boundary F-score measures delineation accuracy independent of detection.',
            font=dict(size=12, family=FONT_FAMILY, color='#666'),
        )],
    )
    fig_v7.write_image(os.path.join(OUT_DIR, 'V7_boundary_fscore.png'), scale=2)
    apply_ppt_style(fig_v7)
    fig_v7.show()
    print('Saved: V7_boundary_fscore.png')


/tmp/ipykernel_8872/547620098.py:33: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig_v7.write_image(os.path.join(OUT_DIR, 'V7_boundary_fscore.png'), scale=2)


Saved: V7_boundary_fscore.png


In [30]:
# ── R1: Overall macro F1 — raster ─────────────────────────────────────────
RAS_ORDER = [
    'wsf_tracker@10m', 'obt_2023@10m',
    'ghsl_built_s_2025@100m', 'tempo_2023q4@100m',
    'obt_2023@100m', 'wsf_tracker@100m',
]
_r = ras[ras['ds_key'].isin(RAS_ORDER)].copy()
_stats_r = (_r.groupby('ds_key')['f1_tile_mean']
             .agg(mean='mean', sd='std').reindex(RAS_ORDER)
             .reset_index())
# Sort ascending for horizontal bar
_stats_r = _stats_r.sort_values('mean', ascending=True)

fig_r1 = go.Figure()
fig_r1.add_trace(go.Bar(
    y=[DATASET_LABELS.get(d, d) for d in _stats_r['ds_key']],
    x=_stats_r['mean'],
    orientation='h',
    marker_color=[DATASET_COLORS.get(d, '#888') for d in _stats_r['ds_key']],
    error_x=dict(type='data', array=_stats_r['sd'].fillna(0).tolist(), visible=True, thickness=2),
    text=[f'<b>{v:.3f}</b>' if not (isinstance(v, float) and v != v) else '' for v in _stats_r['mean']],
    textposition='outside',
    textfont=dict(size=13, family=FONT_FAMILY, color=FONT_COLOR),
    cliponaxis=False,
))
apply_ppt_style(fig_r1,
    title='Raster Datasets — Overall Accuracy (Macro F1, mean ± SD across cities)',
    height=550, width=900)
fig_r1.update_layout(xaxis=dict(range=[0, 1.15], title='Macro F1'), yaxis_title='')
fig_r1.write_image(os.path.join(OUT_DIR, 'R1_raster_macro_f1.png'), scale=2)
apply_ppt_style(fig_r1)
fig_r1.show()
print('Saved: R1_raster_macro_f1.png')


/tmp/ipykernel_8872/2881307983.py:30: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig_r1.write_image(os.path.join(OUT_DIR, 'R1_raster_macro_f1.png'), scale=2)


Saved: R1_raster_macro_f1.png


In [31]:
# ── R2: Dataset wins — raster ─────────────────────────────────────────────
SIX = set(RAS_ORDER)
_city_cov_r = ras.groupby('city')['ds_key'].apply(set)
_cities_all6 = _city_cov_r[_city_cov_r.apply(lambda s: SIX.issubset(s))].index
ras6 = ras[ras['city'].isin(_cities_all6) & ras['ds_key'].isin(RAS_ORDER)].copy()
print(f'Cities with all 6 raster ds_keys: {len(_cities_all6)}')

_rank_counts_r = {ds: {1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0} for ds in RAS_ORDER}
for city, grp in ras6.groupby('city'):
    ranked = grp.sort_values('f1_tile_mean', ascending=False).reset_index(drop=True)
    for ri, row in ranked.iterrows():
        dk = row['ds_key']
        if dk in _rank_counts_r:
            _rank_counts_r[dk][ri + 1] = _rank_counts_r[dk].get(ri + 1, 0) + 1

fig_r2 = go.Figure()
_opacities_r = {1: 1.0, 2: 0.75, 3: 0.55, 4: 0.40, 5: 0.28, 6: 0.18}
for rank in range(6, 0, -1):
    _vals = [_rank_counts_r[d].get(rank, 0) for d in RAS_ORDER]
    if sum(_vals) == 0: continue
    fig_r2.add_trace(go.Bar(
        y=[DATASET_LABELS.get(d, d) for d in RAS_ORDER],
        x=_vals,
        name=f'Rank {rank}',
        orientation='h',
        marker_color=[DATASET_COLORS.get(d, '#888') for d in RAS_ORDER],
        opacity=_opacities_r[rank],
    ))
for dk in RAS_ORDER:
    r1 = _rank_counts_r[dk].get(1, 0)
    fig_r2.add_annotation(
        y=DATASET_LABELS.get(dk, dk), x=len(_cities_all6) + 1,
        text=f'{r1} wins', showarrow=False,
        font=dict(size=11, family=FONT_FAMILY, color=DATASET_COLORS.get(dk, '#333')),
    )
apply_ppt_style(fig_r2,
    title=f'Which Raster Dataset Wins? (n={len(_cities_all6)} cities, all datasets present)',
    height=500, width=1000)
fig_r2.update_layout(barmode='stack', xaxis_title='Number of Cities', yaxis_title='')
fig_r2.write_image(os.path.join(OUT_DIR, 'R2_raster_wins.png'), scale=2)
apply_ppt_style(fig_r2)
fig_r2.show()
print('Saved: R2_raster_wins.png')


Cities with all 6 raster ds_keys: 89


/tmp/ipykernel_8872/1664797162.py:40: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig_r2.write_image(os.path.join(OUT_DIR, 'R2_raster_wins.png'), scale=2)


Saved: R2_raster_wins.png


In [32]:
# ── R3: Complementary metrics — raster ────────────────────────────────────
_r3 = ras[ras['ds_key'].isin(RAS_ORDER)].copy()
_metrics_r3 = [
    ('f1_tile_mean',       'Macro F1 (tile mean)',    [0, 1.15],  None),
    ('signed_area_bias',   'Signed Area Bias (mean)', None,       0),
    ('rel_area_error_mean','Rel. Area Error (mean)',  None,       0),
]
fig_r3 = make_subplots(rows=1, cols=3,
    subplot_titles=[m[1] for m in _metrics_r3],
    shared_yaxes=True, horizontal_spacing=0.06)

for col_idx, (col, label, x_range, vline) in enumerate(_metrics_r3, 1):
    if col not in _r3.columns:
        print(f'[R3] missing column: {col}')
        continue
    _stats_r3 = _r3.groupby('ds_key')[col].agg(mean='mean', sd='std').reindex(RAS_ORDER)
    fig_r3.add_trace(go.Bar(
        y=[DATASET_LABELS.get(d, d) for d in RAS_ORDER],
        x=_stats_r3['mean'].values,
        orientation='h',
        marker_color=[DATASET_COLORS.get(d, '#888') for d in RAS_ORDER],
        error_x=dict(type='data', array=_stats_r3['sd'].fillna(0).tolist(), visible=True, thickness=1.5),
        showlegend=False,
        text=[f'{v:.3f}' if not (isinstance(v, float) and v != v) else '' for v in _stats_r3['mean']],
        textposition='outside',
        textfont=dict(size=11, family=FONT_FAMILY, color=FONT_COLOR),
        cliponaxis=False,
    ), row=1, col=col_idx)
    if vline is not None:
        fig_r3.add_vline(x=vline, line_dash='dash', line_color='gray', opacity=0.5, row=1, col=col_idx)
    if x_range:
        fig_r3.update_xaxes(range=x_range, row=1, col=col_idx)

apply_ppt_style(fig_r3,
    title='Raster Accuracy: F1, Area Bias, and Relative Area Error',
    height=550, width=1400)
fig_r3.write_image(os.path.join(OUT_DIR, 'R3_complementary_metrics_raster.png'), scale=2)
apply_ppt_style(fig_r3)
fig_r3.show()
print('Saved: R3_complementary_metrics_raster.png')


/tmp/ipykernel_8872/3958994974.py:37: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig_r3.write_image(os.path.join(OUT_DIR, 'R3_complementary_metrics_raster.png'), scale=2)


Saved: R3_complementary_metrics_raster.png


In [33]:
# ── R4: 10m vs 100m resolution comparison ─────────────────────────────────
if 'resolution_m' not in ras.columns:
    ras['resolution_m'] = ras['ds_key'].apply(lambda k: 10 if '10m' in str(k) else 100)

_r4 = ras[ras['ds_key'].isin(RAS_ORDER)].dropna(subset=['f1_tile_mean']).copy()
_r4['res_label'] = _r4['resolution_m'].apply(lambda r: '10m' if r == 10 else '100m')
_res_colors = {'10m': '#C94A35', '100m': '#7B4EBD'}

fig_r4 = go.Figure()
for res_lab in ['10m', '100m']:
    _d = _r4[_r4['res_label'] == res_lab]
    fig_r4.add_trace(go.Violin(
        y=_d['f1_tile_mean'], name=res_lab,
        line=dict(color=_res_colors[res_lab], width=2),
        fillcolor=_res_colors[res_lab], opacity=0.5,
        meanline_visible=True, box_visible=True,
        points='all', pointpos=0, jitter=0.35,
        marker=dict(color=_res_colors[res_lab], size=5, opacity=0.6),
        text=(_d['city'] + ' / ' + _d['ds_key']).tolist(),
        hovertemplate='<b>%{text}</b><br>F1: %{y:.4f}<extra></extra>',
    ))
apply_ppt_style(fig_r4,
    title='Effect of Resolution: 10m vs 100m Raster Datasets (macro F1 per city)',
    height=650, width=800)
fig_r4.update_layout(
    yaxis=dict(title='Macro F1 (tile mean)', range=[-0.05, 1.08]),
    annotations=[dict(
        x=0.5, y=-0.12, xref='paper', yref='paper', showarrow=False,
        text='Higher resolution does not improve F1 — 100m tasks are coarser and easier to match.',
        font=dict(size=12, family=FONT_FAMILY, color='#555'),
    )],
)
fig_r4.write_image(os.path.join(OUT_DIR, 'R4_resolution_comparison.png'), scale=2)
apply_ppt_style(fig_r4)
fig_r4.show()
print('Saved: R4_resolution_comparison.png')


/tmp/ipykernel_8872/3243754602.py:33: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig_r4.write_image(os.path.join(OUT_DIR, 'R4_resolution_comparison.png'), scale=2)


Saved: R4_resolution_comparison.png


In [34]:
# ── S1 & S2: IoU threshold comparison ─────────────────────────────────────
if iou.empty:
    print('[S1/S2] iou DataFrame is empty — skipping')
else:
    # Expected columns: city, dataset, iou_threshold, f1, is_spacenet7
    for _c in ['f1', 'iou_threshold']:
        if _c in iou.columns:
            iou[_c] = pd.to_numeric(iou[_c], errors='coerce')

    _iou_sn7 = iou[iou.get('is_spacenet7', pd.Series(True, index=iou.index)) == True] if 'is_spacenet7' in iou.columns else iou
    _iou_ds = sorted(_iou_sn7['dataset'].dropna().unique()) if 'dataset' in _iou_sn7.columns else []
    _iou_thresholds = sorted(_iou_sn7['iou_threshold'].dropna().unique()) if 'iou_threshold' in _iou_sn7.columns else []

    if len(_iou_thresholds) >= 2 and len(_iou_ds) > 0:
        _t_hi = max(_iou_thresholds)
        _t_lo = min(_iou_thresholds)

        fig_s1 = make_subplots(rows=1, cols=2,
            subplot_titles=[f'By Dataset (SpaceNet7 cities)', f'Pooled across datasets'],
            horizontal_spacing=0.12)

        for ds in _iou_ds:
            for thr, opacity in [(_t_lo, 1.0), (_t_hi, 0.5)]:
                _d = _iou_sn7[(_iou_sn7['dataset'] == ds) & (_iou_sn7['iou_threshold'] == thr)]
                _mean = _d['f1'].mean()
                _sd   = _d['f1'].std()
                fig_s1.add_trace(go.Bar(
                    x=[DATASET_LABELS.get(ds, ds)],
                    y=[_mean],
                    name=f'IoU={thr}',
                    marker_color=DATASET_COLORS.get(ds, '#888'),
                    opacity=opacity,
                    error_y=dict(type='data', array=[_sd], visible=True, thickness=1.5),
                    showlegend=(ds == _iou_ds[0]),
                ), row=1, col=1)

        for thr, opacity in [(_t_lo, 1.0), (_t_hi, 0.5)]:
            _pool = _iou_sn7[_iou_sn7['iou_threshold'] == thr]['f1']
            fig_s1.add_trace(go.Bar(
                x=[f'IoU={thr}'], y=[_pool.mean()],
                name=f'IoU={thr} pooled',
                marker_color='#555' if thr == _t_lo else '#AAA',
                error_y=dict(type='data', array=[_pool.std()], visible=True, thickness=2),
                showlegend=False,
            ), row=1, col=2)

        apply_ppt_style(fig_s1,
            title='Effect of IoU Threshold on F1 Score (SpaceNet7 cities only, macro F1 ± SD)',
            height=600, width=1100)
        fig_s1.update_layout(barmode='group')
        fig_s1.write_image(os.path.join(OUT_DIR, 'S1_iou_threshold.png'), scale=2)
        apply_ppt_style(fig_s1)
        fig_s1.show()
        print('Saved: S1_iou_threshold.png')

        # ── S2: F1 by source — IoU comparison ─────────────────────────────
        if 'source' not in iou.columns and 'city' in iou.columns and 'source' in vec.columns:
            _src_map = vec[['city', 'source']].drop_duplicates('city').set_index('city')['source']
            iou['source'] = iou['city'].map(_src_map).fillna('Other / Gov')

        if 'source' in iou.columns:
            _src_order = ['SpaceNet7', 'HotOSM', 'Other / Gov']
            fig_s2 = go.Figure()
            for thr, opacity in [(_t_lo, 1.0), (_t_hi, 0.55)]:
                _src_stats_s2 = (iou[iou['iou_threshold'] == thr]
                                 .groupby('source')['f1']
                                 .agg(mean='mean', sd='std')
                                 .reindex(_src_order))
                for src in _src_order:
                    if src not in _src_stats_s2.index: continue
                    _mean_s2 = _src_stats_s2.loc[src, 'mean']
                    _sd_s2   = _src_stats_s2.loc[src, 'sd']
                    fig_s2.add_trace(go.Bar(
                        x=[src], y=[_mean_s2],
                        name=f'IoU={thr}',
                        marker_color=SOURCE_COLORS.get(src, '#888'),
                        opacity=opacity,
                        error_y=dict(type='data', array=[_sd_s2], visible=True, thickness=1.5),
                        showlegend=(src == _src_order[0]),
                    ))
            apply_ppt_style(fig_s2,
                title='IoU Threshold Effect by Reference Data Source',
                height=600, width=900)
            fig_s2.update_layout(barmode='group',
                xaxis_title='Reference Data Source', yaxis_title='F1')
            fig_s2.write_image(os.path.join(OUT_DIR, 'S2_iou_by_source.png'), scale=2)
            apply_ppt_style(fig_s2)
            fig_s2.show()
            print('Saved: S2_iou_by_source.png')
    else:
        print(f'[S1/S2] need >= 2 IoU thresholds and datasets; found thresholds={_iou_thresholds}, datasets={len(_iou_ds)}')


[S1/S2] iou DataFrame is empty — skipping


In [35]:
# ── W1: WSF F1 vs reference year ──────────────────────────────────────────
if wsf_sweep.empty:
    print('[W1] wsf_sweep is empty — skipping')
else:
    for _c in ['as_of_year', 'mean_f1', 'mean_precision', 'mean_recall', 'f1_delta_vs_baseline']:
        if _c in wsf_sweep.columns:
            wsf_sweep[_c] = pd.to_numeric(wsf_sweep[_c], errors='coerce')
    wsf_sweep = wsf_sweep.sort_values('as_of_year')

    _baseline_mask = wsf_sweep['f1_delta_vs_baseline'].abs() < 1e-9 if 'f1_delta_vs_baseline' in wsf_sweep.columns else pd.Series(False, index=wsf_sweep.index)
    _baseline_year = float(wsf_sweep.loc[_baseline_mask, 'as_of_year'].iloc[0]) if _baseline_mask.any() else None
    _peak_idx = wsf_sweep['mean_f1'].idxmax()
    _peak_year = float(wsf_sweep.loc[_peak_idx, 'as_of_year'])
    _peak_f1   = float(wsf_sweep.loc[_peak_idx, 'mean_f1'])

    fig_w1 = go.Figure()
    fig_w1.add_trace(go.Scatter(
        x=wsf_sweep['as_of_year'], y=wsf_sweep['mean_f1'],
        mode='lines+markers', name='Mean F1',
        line=dict(color='#C94A35', width=2.5),
        marker=dict(size=7, color='#C94A35'),
    ))
    if 'mean_precision' in wsf_sweep.columns:
        fig_w1.add_trace(go.Scatter(
            x=wsf_sweep['as_of_year'], y=wsf_sweep['mean_precision'],
            mode='lines', name='Mean Precision',
            line=dict(color='#C94A35', width=1.5, dash='dash'), opacity=0.6,
        ))
    if 'mean_recall' in wsf_sweep.columns:
        fig_w1.add_trace(go.Scatter(
            x=wsf_sweep['as_of_year'], y=wsf_sweep['mean_recall'],
            mode='lines', name='Mean Recall',
            line=dict(color='#C94A35', width=1.5, dash='dot'), opacity=0.6,
        ))
    if _baseline_year:
        fig_w1.add_vline(x=_baseline_year, line_dash='dash', line_color='#555', opacity=0.5,
                         annotation_text='baseline', annotation_position='top right')
    fig_w1.add_annotation(
        x=_peak_year, y=_peak_f1,
        text=f'Peak: F1={_peak_f1:.3f} ({_peak_year})',
        arrowhead=2, showarrow=True, arrowcolor='#333',
        font=dict(size=12, family=FONT_FAMILY, color='#333'),
        yshift=12,
    )
    apply_ppt_style(fig_w1,
        title='WSF Tracker Accuracy by Reference Year (year-code sensitivity sweep)',
        height=600, width=1100)
    fig_w1.update_layout(xaxis_title='Reference Year', yaxis_title='Score', yaxis=dict(range=[0, 1.05]))
    fig_w1.write_image(os.path.join(OUT_DIR, 'W1_wsf_year_sensitivity.png'), scale=2)
    apply_ppt_style(fig_w1)
    fig_w1.show()
    print('Saved: W1_wsf_year_sensitivity.png')

# ── W2: Temporal alignment improvement ────────────────────────────────────
if wsf_aligned.empty:
    print('[W2] wsf_aligned is empty — skipping')
else:
    for _c in ['f1_baseline', 'f1_aligned', 'f1_delta']:
        if _c in wsf_aligned.columns:
            wsf_aligned[_c] = pd.to_numeric(wsf_aligned[_c], errors='coerce')
    if 'is_spacenet7' in wsf_aligned.columns:
        wsf_aligned['is_spacenet7'] = wsf_aligned['is_spacenet7'].astype(bool)

    fig_w2 = make_subplots(rows=1, cols=2,
        subplot_titles=['F1: Baseline vs Year-Aligned (per city)', 'Mean F1 Delta by Group'],
        horizontal_spacing=0.12)

    # Left: scatter
    for _is_sn7, _label, _color in [(True, 'SpaceNet7', '#E05C00'), (False, 'Non-SpaceNet7', '#0072B2')]:
        if 'is_spacenet7' in wsf_aligned.columns:
            _d = wsf_aligned[wsf_aligned['is_spacenet7'] == _is_sn7].dropna(subset=['f1_baseline','f1_aligned'])
        else:
            _d = wsf_aligned.dropna(subset=['f1_baseline','f1_aligned'])
            _label = 'All cities'
        if _d.empty: continue
        fig_w2.add_trace(go.Scatter(
            x=_d['f1_baseline'], y=_d['f1_aligned'],
            mode='markers', name=_label,
            marker=dict(color=_color, size=7, opacity=0.75),
            text=_d['city'].tolist() if 'city' in _d.columns else [],
            hovertemplate='<b>%{text}</b><br>Baseline: %{x:.3f}<br>Aligned: %{y:.3f}<extra></extra>',
        ), row=1, col=1)
    # Diagonal y=x
    fig_w2.add_trace(go.Scatter(
        x=[0, 1], y=[0, 1], mode='lines', showlegend=False,
        line=dict(color='gray', width=1, dash='dot'),
    ), row=1, col=1)

    # Right: bar chart of mean delta by group
    _groups_w2 = []
    if 'is_spacenet7' in wsf_aligned.columns and 'f1_delta' in wsf_aligned.columns:
        for _is_sn7, _label in [(True, 'SpaceNet7'), (False, 'Non-SpaceNet7')]:
            _d = wsf_aligned[wsf_aligned['is_spacenet7'] == _is_sn7]['f1_delta'].dropna()
            _groups_w2.append((_label, float(_d.mean()), _d.std()))
        _all_delta = wsf_aligned['f1_delta'].dropna()
        _groups_w2.append(('All', float(_all_delta.mean()), _all_delta.std()))
    elif 'f1_delta' in wsf_aligned.columns:
        _all_delta = wsf_aligned['f1_delta'].dropna()
        _groups_w2 = [('All cities', float(_all_delta.mean()), _all_delta.std())]

    for _label, _mean_d, _sd_d in _groups_w2:
        _color = '#E05C00' if 'Space' in _label else ('#0072B2' if 'Non' in _label else '#555')
        fig_w2.add_trace(go.Bar(
            x=[_label], y=[_mean_d],
            name=_label,
            marker_color=_color,
            error_y=dict(type='data', array=[_sd_d], visible=True, thickness=2),
            text=[f'{_mean_d:+.3f}'],
            textposition='outside',
            textfont=dict(size=13, family=FONT_FAMILY, color=FONT_COLOR),
            showlegend=False,
        ), row=1, col=2)
    fig_w2.add_hline(y=0, line_color='#333', line_width=1.5, row=1, col=2)

    apply_ppt_style(fig_w2,
        title='Per-City Temporal Alignment Improves WSF Tracker Accuracy',
        height=600, width=1200)
    fig_w2.update_xaxes(title_text='F1 (pipeline baseline)', row=1, col=1)
    fig_w2.update_yaxes(title_text='F1 (year-aligned to reference)', row=1, col=1)
    fig_w2.update_yaxes(title_text='Mean F1 Delta', row=1, col=2)
    fig_w2.write_image(os.path.join(OUT_DIR, 'W2_wsf_alignment.png'), scale=2)
    apply_ppt_style(fig_w2)
    fig_w2.show()
    print('Saved: W2_wsf_alignment.png')


[W1] wsf_sweep is empty — skipping
[W2] wsf_aligned is empty — skipping


In [36]:
# ── Summary of saved figures ───────────────────────────────────────────────
_figures = [
    ('V1_vector_macro_f1.png',          'Vector macro F1: overall mean ± SD per dataset'),
    ('V2_f1_by_source.png',             'Vector F1 broken out by reference data source'),
    ('V3_vector_wins.png',              'Stacked bar: which vector dataset ranks best per city'),
    ('V4_complementary_metrics_vector.png', 'Vector: F1, total area bias, count ratio side-by-side'),
    ('V5_f1_vs_bias_vector.png',        'Scatter: F1 vs total area bias (city level)'),
    ('V6_precision_recall.png',         'Scatter: precision vs recall with F1 iso-curves'),
    ('V7_boundary_fscore.png',          'Boundary F-score per vector dataset'),
    ('R1_raster_macro_f1.png',          'Raster macro F1: overall mean ± SD per ds_key'),
    ('R2_raster_wins.png',              'Stacked bar: which raster dataset ranks best per city'),
    ('R3_complementary_metrics_raster.png', 'Raster: F1, signed area bias, rel. area error'),
    ('R4_resolution_comparison.png',    'Violin: 10m vs 100m raster F1 distribution'),
    ('S1_iou_threshold.png',            'IoU threshold effect on F1 (SpaceNet7, per dataset + pooled)'),
    ('S2_iou_by_source.png',            'IoU threshold effect by reference data source'),
    ('W1_wsf_year_sensitivity.png',     'WSF F1 vs reference year sweep'),
    ('W2_wsf_alignment.png',            'WSF temporal alignment improvement: scatter + delta bar'),
]

print(f'{"Figure":<45} Description')
print('-' * 100)
for fname, desc in _figures:
    _exists = '✓' if os.path.exists(os.path.join(OUT_DIR, fname)) else '○'
    print(f'{_exists}  {fname:<43} {desc}')
print(f'\nFigures saved to: {OUT_DIR}')


Figure                                        Description
----------------------------------------------------------------------------------------------------
✓  V1_vector_macro_f1.png                      Vector macro F1: overall mean ± SD per dataset
✓  V2_f1_by_source.png                         Vector F1 broken out by reference data source
✓  V3_vector_wins.png                          Stacked bar: which vector dataset ranks best per city
✓  V4_complementary_metrics_vector.png         Vector: F1, total area bias, count ratio side-by-side
✓  V5_f1_vs_bias_vector.png                    Scatter: F1 vs total area bias (city level)
✓  V6_precision_recall.png                     Scatter: precision vs recall with F1 iso-curves
✓  V7_boundary_fscore.png                      Boundary F-score per vector dataset
✓  R1_raster_macro_f1.png                      Raster macro F1: overall mean ± SD per ds_key
✓  R2_raster_wins.png                          Stacked bar: which raster dataset ranks bes